# Stage 5: quantum point contact and channel filtering

This notebook examines a smooth gate constriction in the Stage 4 finite-width strip. It reuses the Stage 4 Hamiltonian, mode, semi-infinite lead, Green-function, broadening and Landauer routines, and the Stage 2 site-resolved LDOS function. The model is coherent and at zero temperature; the reported LDOS is not normalized.

In [ ]:
from pathlib import Path
import csv
import json
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "results/stage5_qpc_transmission.csv").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from quantum_transport.density import local_density_of_states
from quantum_transport.hamiltonian import tight_binding_hamiltonian
from quantum_transport.modes import open_mode_count, transverse_modes
from quantum_transport.qpc import quantum_point_contact_potential
from quantum_transport.two_dimensional import rectangular_device_hamiltonian
from quantum_transport.two_dimensional_leads import two_dimensional_self_energy_matrices
from quantum_transport.two_dimensional_transport import (
    broadening_matrix, retarded_2d_device_green_function, multichannel_transmission,
    physical_conductance,
)

def read_csv(name):
    with (ROOT / "results" / name).open(newline="") as stream:
        return list(csv.DictReader(stream))

metadata = json.loads((ROOT / "results/stage5_qpc_metadata.json").read_text())
transmission_rows = read_csv("stage5_qpc_transmission.csv")
scan_rows = read_csv("stage5_qpc_gate_scan.csv")
Lx, Ly, t, epsilon_0, t_c = 40, 6, 1.0, 0.0, 1.0
print("QPC metadata:", metadata["qpc_potential"])
print("saved transmission rows:", len(transmission_rows))

## 1. Clean 2D reference

First compare the clean device with the analytic incoming lead mode count at energies away from transverse thresholds. The gate-zero profile below is identically the clean onsite background.

In [ ]:
lead_modes, lead_vectors = transverse_modes(Ly, epsilon_0, t)
print("lead transverse energies:", lead_modes)
for energy in (-3.5, -3.0, -2.2, -1.3):
    clean_potential = quantum_point_contact_potential(Lx, Ly, 0.0)
    clean_H = rectangular_device_hamiltonian(Lx, Ly, clean_potential, t)
    sigma_L, sigma_R = two_dimensional_self_energy_matrices(energy, Lx, Ly, epsilon_0, t, t_c)
    clean_G = retarded_2d_device_green_function(energy, clean_H, sigma_L, sigma_R)
    clean_T = multichannel_transmission(clean_G, broadening_matrix(sigma_L), broadening_matrix(sigma_R))
    print(f"E={energy:g}: N_open={open_mode_count(energy, lead_modes, t)}, T={clean_T:.12g}")

## 2. Smooth QPC potential

The implemented gate profile is `epsilon_0 + Vg exp(-(x-xc)^2/(2 sigma_x^2)) [1 + alpha ((y-yc)/y_scale)^2]`. It is mirror-symmetric in x and y, approaches the background away from the centre, and raises the off-centre rows more strongly.

In [ ]:
potential_rows = read_csv("stage5_qpc_potential.csv")
V = np.zeros((Lx, Ly))
for row in potential_rows:
    V[int(row["x"]), int(row["y"])] = float(row["onsite_energy"])
fig, ax = plt.subplots(figsize=(8, 3.5))
im = ax.imshow(V.T, origin="lower", aspect="auto", cmap="magma")
ax.set(xlabel="Longitudinal slice x", ylabel="Transverse site y", title="QPC onsite potential, strongest gate")
fig.colorbar(im, ax=ax, label="Onsite energy (t units)")
plt.tight_layout()
plt.show()

## 3?4. Transmission and conductance

The dimensionless conductance is `g=T`; physical conductance is `G=(2e^2/h)T`. The saved curves were calculated without clipping to the open-channel count.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for gate in metadata["qpc_potential"]["gate_strengths"]:
    selected = [r for r in transmission_rows if float(r["gate_strength"]) == gate]
    ax.plot([float(r["energy"]) for r in selected],
            [float(r["transmission_T"]) for r in selected], label=f"Vg={gate:g}")
ax.set(xlabel="Energy (tight-binding units)", ylabel="T(E)", title="Transmission through the QPC")
ax.grid(alpha=0.25); ax.legend(ncol=2); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8, 4))
for gate in metadata["qpc_potential"]["gate_strengths"]:
    selected = [r for r in transmission_rows if float(r["gate_strength"]) == gate]
    ax.plot([float(r["energy"]) for r in selected],
            [float(r["conductance_S"])*1e6 for r in selected], label=f"Vg={gate:g}")
ax.set(xlabel="Energy (tight-binding units)", ylabel="G (microSiemens)", title="Physical conductance")
ax.grid(alpha=0.25); ax.legend(ncol=2); plt.tight_layout(); plt.show()

## 5. Gate-voltage dependence

At selected fixed energies, compare transmission with increasing gate strength. The lead `N_open` remains fixed as the gate changes; central slice modes provide a separate local energetic indicator.

In [ ]:
for energy in (-3.5, -3.0, -2.5, -2.0, -1.5):
    selected = [r for r in scan_rows if abs(float(r["energy"])-energy) < 1e-12]
    print(f"E={energy:g}, N_open={int(selected[0]['N_open_lead'])}: " +
          ", ".join(f"Vg={float(r['gate_strength']):g}: T={float(r['transmission_T']):.5g}, "
                    f"Nlocal={r['N_local_open_centre']}" for r in selected))
fig, ax = plt.subplots(figsize=(8, 4))
for energy in (-3.5, -3.0, -2.5, -2.0, -1.5):
    selected = [r for r in scan_rows if abs(float(r["energy"])-energy) < 1e-12]
    ax.plot([float(r["gate_strength"]) for r in selected],
            [float(r["transmission_T"]) for r in selected], marker="o", label=f"E={energy:g}")
ax.set(xlabel="Gate strength Vg", ylabel="Transmission T", title="Channel filtering versus gate")
ax.grid(alpha=0.25); ax.legend(ncol=2); plt.tight_layout(); plt.show()

## 6. Transverse mode interpretation

The analytic lead mode thresholds come from the open transverse slice. For each gate, the eigenvalues of the central transverse slice are used to count modes satisfying `|E-epsilon_local,n| <= 2t`. This local count is not an exact conserved channel bound in a finite, longitudinally varying constriction; evanescent tunnelling can yield nonzero transmission even where the local count is zero.

In [ ]:
for gate, info in metadata["local_centre_subbands_by_gate"].items():
    print(f"Vg={float(gate):g}: central transverse energies=" +
          ", ".join(f"{value:.5f}" for value in info["transverse_energies"]))
energy = -2.0
gate = 0.75
V_center = quantum_point_contact_potential(Lx, Ly, gate,
    longitudinal_sigma=metadata["qpc_potential"]["longitudinal_sigma_sites"],
    transverse_strength=metadata["qpc_potential"]["transverse_strength_alpha"])
H_y_center = tight_binding_hamiltonian(Ly, epsilon_0 + V_center[(Lx-1)//2], t)
local_modes = np.linalg.eigvalsh(H_y_center)
print("At E=-2 and Vg=.75: N_open lead =", open_mode_count(energy, lead_modes, t),
      "; local central open-mode indicator =", open_mode_count(energy, local_modes, t))

## 7. Spatial LDOS

The maps use `rho_i(E)=-Im(G^r_ii)/pi` from the Stage 2 function. Raw values are saved and plotted with a shared colour scale so changes remain visible; no LDOS normalization is applied.

In [ ]:
ldos_rows = read_csv("stage5_qpc_ldos.csv")
ldos_energy = float(metadata["ldos"]["energy"])
figure, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
rho_by_gate = {}
for ax, gate in zip(axes, metadata["ldos"]["gate_strengths"]):
    profile = quantum_point_contact_potential(
        Lx, Ly, gate,
        longitudinal_sigma=metadata["qpc_potential"]["longitudinal_sigma_sites"],
        transverse_strength=metadata["qpc_potential"]["transverse_strength_alpha"],
    )
    H = rectangular_device_hamiltonian(Lx, Ly, profile, t)
    sigma_L, sigma_R = two_dimensional_self_energy_matrices(ldos_energy, Lx, Ly, epsilon_0, t, t_c)
    G = retarded_2d_device_green_function(ldos_energy, H, sigma_L, sigma_R)
    rho = local_density_of_states(G).reshape(Lx, Ly)
    rho_by_gate[gate] = rho
    ax.imshow(rho.T, origin="lower", aspect="auto", cmap="viridis", vmin=0,
              vmax=max(float(r["ldos_per_energy"]) for r in ldos_rows))
    ax.set(xlabel="Longitudinal slice x", title=f"Vg={gate:g}")
axes[0].set_ylabel("Transverse site y")
figure.suptitle(f"Raw LDOS at E={ldos_energy:g}")
figure.colorbar(axes[0].images[0], ax=axes, label="LDOS per energy")
plt.show()
for gate, rho in rho_by_gate.items():
    saved = np.zeros((Lx, Ly))
    for row in ldos_rows:
        if float(row["gate_strength"]) == gate:
            saved[int(row["x"]), int(row["y"])] = float(row["ldos_per_energy"])
    np.testing.assert_allclose(rho, saved, atol=1e-13)